In [1]:
import sympy as sym
import numpy as np
import dill
import time
import random
from scipy.linalg import null_space
from IPython.display import Math, display

import importlib
import esekf_definitions

# since I often change the definitions while I work in this file
importlib.reload(esekf_definitions)
from esekf_definitions import *

sym.init_printing(use_unicode=True)

print(np.__version__)

2.5.2


---

# Intro

## Fixed Values

For each cycle, we calculate $\Delta t$ in code and we assume a fixed gravity acceleration force. For the offset between IMU and camera, we define translation and rotation.

\begin{align*}
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}


In [2]:
dt = sym.Symbol(r"\Delta t", real=True, positive=True)
subs_dt = {
    dt: 0.01
}  # approx 100 Hz sampling rate (not actually fixed in the system but fixed per discrete time step)

gravity = sym.Symbol(r"g", real=True, positive=True)
gravity_vector_sym = sym.Matrix([0, 0, -1 * gravity])
subs_gravity = {gravity: gravity_constant}

# \Delta p_{I \to c}
IMU_cam_x_translation = 0
IMU_cam_y_translation = sym.Symbol(r"\Delta{p}_y", negative=True)
IMU_cam_z_translation = sym.Symbol(r"\Delta{p}_z", negative=True)
IMU_cam_translation = sym.Matrix(
    [IMU_cam_x_translation, IMU_cam_y_translation, IMU_cam_z_translation]
)

subs_IMU_cam_translation = {
    IMU_cam_y_translation: IMU_cam_translation_value[1],
    IMU_cam_z_translation: IMU_cam_translation_value[2],
}

# \Delta q_{I \to c}
IMU_cam_w_rotation = sym.Symbol(r"\Delta{q}_w", real=True)
IMU_cam_x_rotation = sym.Symbol(r"\Delta{q}_x", real=True)
IMU_cam_y_rotation = 0
IMU_cam_z_rotation = 0
IMU_cam_rotation = sym.Matrix(
    [IMU_cam_w_rotation, IMU_cam_x_rotation, IMU_cam_y_rotation, IMU_cam_z_rotation]
)

subs_IMU_cam_rotation = {
    IMU_cam_w_rotation: IMU_cam_rotation_value[0],
    IMU_cam_x_rotation: IMU_cam_rotation_value[1],
}

subs_fixed_values = (
    subs_dt | subs_gravity | subs_IMU_cam_rotation | subs_IMU_cam_translation
)

# how substitution works later, .subs() does not overwrite my sympy expressions
dt.subs(subs_dt), gravity.subs(subs_gravity), IMU_cam_translation.subs(
    subs_IMU_cam_translation
), IMU_cam_rotation.subs(subs_IMU_cam_rotation)

⎛                           ⎡0.629320391049838 ⎤⎞
⎜               ⎡   0    ⎤  ⎢                  ⎥⎟
⎜               ⎢        ⎥  ⎢-0.777145961456971⎥⎟
⎜0.01, 9.80665, ⎢-0.0013 ⎥, ⎢                  ⎥⎟
⎜               ⎢        ⎥  ⎢        0         ⎥⎟
⎜               ⎣-0.00662⎦  ⎢                  ⎥⎟
⎝                           ⎣        0         ⎦⎠

---

# Input

\begin{align*}
&\text{Accelerometer input: } &\tilde a && &\text{noise: } &\tilde a_n \\
&\text{Gyroscope input: } &\tilde \omega && &\text{noise: } &\tilde \omega_n\\
\end{align*}


In [3]:
u_tilde = get_definition("IMU input")["sympy array"]
u_tilde_n = get_definition("IMU input noise")["sympy array"]

a_tilde = get_sym_subarray("a_tilde")
a_tilde_n = get_sym_subarray("a_tilde_n")

omega_tilde = get_sym_subarray("omega_tilde")
omega_tilde_n = get_sym_subarray("omega_tilde_n")

u_tilde.T, u_tilde_n.T

([\tilde{a}_{x}  \tilde{a}_{y}  \tilde{a}_{z}  \tilde{\omega}_{x}  \tilde{\ome ↪

↪ ga}_{y}  \tilde{\omega}_{z}], [\tilde{a}_{n,x}  \tilde{a}_{n,y}  \tilde{a}_{ ↪

↪ n,z}  \tilde{\omega}_{n,x}  \tilde{\omega}_{n,y}  \tilde{\omega}_{n,z}])

---

# Nominal State

## IMU

\begin{align*}
x_\text{IMU} = \begin{bmatrix} p & v & q & a_b & \omega_b \end{bmatrix}^\top \qquad \qquad x \in SO(3) \times \left(\mathbb{R}^3\right)^4
\end{align*}


In [4]:
x_IMU = get_definition("IMU nominal state")["sympy array"]

p = get_sym_subarray("p")
v = get_sym_subarray("v")
q = get_sym_subarray("q")
a_b = get_sym_subarray("a_b")
omega_b = get_sym_subarray("omega_b")

x_IMU.T

[p_{x}  p_{y}  p_{z}  v_{x}  v_{y}  v_{z}  q_{w}  q_{x}  q_{y}  q_{z}  a_{b,x} ↪

↪   a_{b,y}  a_{b,z}  \omega_{b,x}  \omega_{b,y}  \omega_{b,z}]

### Kinematics

\begin{align}
\dot p &= v & \dot a_b &= 0 \\
\text{Continuous Case} \qquad \dot v &= \mathbf{R}^{\{q\}}(\tilde a - a_b) + g & \dot \omega_b &= 0 \\
\dot q &= \tfrac{1}{2} q \otimes (\tilde \omega - \omega_b) \\
\\
p &\leftarrow p + v\Delta t + \tfrac{1}{2}( \mathbf{R}^{\{q\}} (\tilde a - a_b) + g)\Delta t^2 \qquad & a_b &\leftarrow a_b \\
\text{Discrete Case} \qquad v &\leftarrow v + (\mathbf{R}^{\{q\}}(\tilde a - a_b) + g)\Delta t & \omega_b &\leftarrow \omega_b \\
q &\leftarrow q \otimes q\{(\tilde \omega - \omega_b) \Delta t\} \\
\end{align} 

In [5]:
# Continuous
p_dot = v
v_dot = R(q) * (a_tilde - a_b) + gravity_vector_sym
q_dot = 0.5 * (left_quat_matrix(q) * sym.Matrix([0, *(omega_tilde - omega_b)]))
a_b_dot = sym.zeros(3, 1)
omega_b_dot = sym.zeros(3, 1)

x_IMU_dot = sym.Matrix.vstack(p_dot, v_dot, q_dot, a_b_dot, omega_b_dot)

# discrete
f_p = p + v * dt + 0.5 * (R(q) * (a_tilde - a_b) + gravity_vector_sym) * dt**2
f_v = v + (R(q) * (a_tilde - a_b) + gravity_vector_sym) * dt
f_q = left_quat_matrix(q) * quat_from_axis_angle((omega_tilde - omega_b) * dt)
f_a_b = a_b
f_omega_b = omega_b

f_x_IMU = sym.Matrix.vstack(f_p, f_v, f_q, f_a_b, f_omega_b)

# f_x_IMU, x_IMU_dot

## Speakers

\begin{align*}
x_s = \begin{bmatrix} p_s & v_s & q_s \end{bmatrix}^\top \qquad \qquad x_s \in SO(3) \times \left(\mathbb{R}^3\right)^2
\end{align*}


In [6]:
x_s = get_definition("Speaker nominal state")["sympy array"]

p_s = get_sym_subarray("p_s")
v_s = get_sym_subarray("v_s")
q_s = get_sym_subarray("q_s")

x_s.T

[p_{s,x}  p_{s,y}  p_{s,z}  v_{s,x}  v_{s,y}  v_{s,z}  q_{s,w}  q_{s,x}  q_{s, ↪

↪ y}  q_{s,z}]

### Kinematics

\begin{align}
\dot p_s &= v_s \\
\text{Continuous Case} \qquad \dot v_s &= 0 \\
\dot q_s &= 0 \\
\\
p_s &\leftarrow p_s + v_s \Delta t \\
\text{Discrete Case}  \qquad v_s &\leftarrow v_s \\
q_s &\leftarrow q_s \\
\end{align} 

In [7]:
p_s_dot = v_s
v_s_dot = sym.zeros(3, 1)
q_s_dot = sym.zeros(4, 1)

x_s_dot = sym.Matrix.vstack(p_s_dot, v_s_dot, q_s_dot)

f_p_s = p_s + v_s * dt
f_v_s = v_s
f_q_s = q_s

f_x_s = sym.Matrix.vstack(f_p_s, f_v_s, f_q_s)
# f_x_s, x_s_dot

## Combined State

\begin{align*}
x = \begin{bmatrix} p & v & q & a_b & \omega_b & p_s & v_s & q_s & p_{s + 1} & v_{s + 1} & q_{s + 1} & \cdots \end{bmatrix}^\top
\end{align*}


In [8]:
x = sym.Matrix.vstack(x_IMU, x_s)
x_dot = sym.Matrix.vstack(x_IMU_dot, x_s_dot)
f_x = sym.Matrix.vstack(f_x_IMU, f_x_s)

# f_x, x_dot

---

# Error State

## IMU

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b \end{bmatrix}^\top \qquad \qquad x \in \left(\mathbb{R}^3\right)^5
\end{align*}


In [9]:
# Full states and process noise fetched from definitions
delta_x_IMU = get_definition("IMU error state")["sympy array"]

delta_p = get_sym_subarray("delta_p")
delta_v = get_sym_subarray("delta_v")
delta_theta = get_sym_subarray("delta_theta")
delta_a_b = get_sym_subarray("delta_a_b")
delta_omega_b = get_sym_subarray("delta_omega_b")

delta_x_n_IMU = get_definition("IMU process noise")["sympy array"]

delta_v_n = get_sym_subarray("delta_v_n")
delta_theta_n = get_sym_subarray("delta_theta_n")
delta_a_b_n = get_sym_subarray("delta_a_b_n")
delta_omega_b_n = get_sym_subarray("delta_omega_b_n")

# Display the transposes
delta_x_IMU.T, delta_x_n_IMU.T

([\delta{p}_{x}  \delta{p}_{y}  \delta{p}_{z}  \delta{v}_{x}  \delta{v}_{y}  \ ↪

↪ delta{v}_{z}  \delta{\theta}_{x}  \delta{\theta}_{y}  \delta{\theta}_{z}  \d ↪

↪ elta{a}_{b,x}  \delta{a}_{b,y}  \delta{a}_{b,z}  \delta{\omega}_{b,x}  \delt ↪

↪ a{\omega}_{b,y}  \delta{\omega}_{b,z}], [\delta{v}_{n,x}  \delta{v}_{n,y}  \ ↪

↪ delta{v}_{n,z}  \delta{\theta}_{n,x}  \delta{\theta}_{n,y}  \delta{\theta}_{ ↪

↪ n,z}  \delta{a}_{bn,x}  \delta{a}_{bn,y}  \delta{a}_{bn,z}  \delta{\omega}_{ ↪

↪ bn,x}  \delta{\omega}_{bn,y}  \delta{\omega}_{bn,z}])

### Kinematics

\begin{align}
\dot{\delta p} &= \delta v & \dot{\delta a_b} &= a_{bn} \\
\text{Continuous Case} \qquad \dot{\delta v} &= -\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b - \mathbf{R}^{\{q\}} \tilde a_n & \dot{\delta \omega_b} &= \omega_{bn} \\
\dot{\delta \theta} &= -[\tilde \omega - \omega_b]_\times \delta \theta - \delta \omega_b - \tilde \omega_n \\
\\
\delta p &\leftarrow \delta p + \delta v\Delta t & \delta a_b &\leftarrow \delta a_b + \delta a_n \\
\text{Discrete Case} \qquad \delta v &\leftarrow \delta v + (-\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b)\Delta t + \delta v_{n} \qquad & \delta \omega_b &\leftarrow \delta \omega_b + \delta \omega_n \\
\delta \theta &\leftarrow (\mathbf{R}^{\{(\tilde \omega - \omega_b) \Delta t\}})^\top \delta \theta - \delta \omega_b \Delta t + \delta \theta_{n}
\end{align}

In [10]:
# Continuous
delta_p_dot = delta_v
delta_v_dot = (
    -R(q) * skew_symmetric_matrix(a_tilde - a_b) * delta_theta
    - R(q) * delta_a_b
    - R(q) * a_tilde_n
)
delta_theta_dot = (
    -1 * skew_symmetric_matrix(omega_tilde - omega_b) * delta_theta
    - delta_omega_b
    - omega_tilde_n
)
# TODO a_b_n existiert noch gar nicht weil ichs bisher nicht brauchte
# ich könnte fürs Erste einfach die noise Werte vom discrete case benutzen, ich will es ja nur in der observability nachweisen
delta_a_b_dot = delta_a_b_n  # technically wrong
delta_omega_b_dot = delta_omega_b_n  # technically wrong

delta_x_IMU_dot = sym.Matrix.vstack(
    delta_p_dot, delta_v_dot, delta_theta_dot, delta_a_b_dot, delta_omega_b_dot
)

# Discrete
f_delta_p = delta_p + delta_v * dt
f_delta_v = (
    delta_v
    + (-R(q) * skew_symmetric_matrix(a_tilde - a_b) * delta_theta - R(q) * delta_a_b)
    * dt
    + delta_v_n
)
f_delta_theta = (
    R(quat_from_axis_angle((omega_tilde - omega_b) * dt)).T * delta_theta
    - delta_omega_b * dt
    + delta_theta_n
)
f_delta_a_b = delta_a_b + delta_a_b_n
f_delta_omega_b = delta_omega_b + delta_omega_b_n

f_delta_x_IMU = sym.Matrix.vstack(
    f_delta_p, f_delta_v, f_delta_theta, f_delta_a_b, f_delta_omega_b
)
# f_delta_x_IMU, delta_x_dot

## Speakers

\begin{align*}
\delta x_s = \begin{bmatrix} \delta p_s & \delta v_s & \delta \theta_s \end{bmatrix}^\top \qquad \qquad x_s \in \left(\mathbb{R}^3\right)^3
\end{align*}


In [11]:
delta_x_s = get_definition("Speaker error state")["sympy array"]

delta_p_s = get_sym_subarray("delta_p_s")
delta_v_s = get_sym_subarray("delta_v_s")
delta_theta_s = get_sym_subarray("delta_theta_s")

delta_x_s_n = get_definition("Speaker process noise")["sympy array"]

delta_v_s_n = get_sym_subarray("delta_v_s_n")
delta_theta_s_n = get_sym_subarray("delta_theta_s_n")


delta_x_s.T, delta_x_s_n.T

([\delta{p}_{s,x}  \delta{p}_{s,y}  \delta{p}_{s,z}  \delta{v}_{s,x}  \delta{v ↪

↪ }_{s,y}  \delta{v}_{s,z}  \delta{\theta}_{s,x}  \delta{\theta}_{s,y}  \delta ↪

↪ {\theta}_{s,z}], [\delta{v}_{sn,x}  \delta{v}_{sn,y}  \delta{v}_{sn,z}  \del ↪

↪ ta{\theta}_{sn,x}  \delta{\theta}_{sn,y}  \delta{\theta}_{sn,z}])

### Kinematics

\begin{align}
\dot{\delta p_s} &= \delta v_s \\
\text{Continuous Case} \qquad \qquad \dot{\delta v_s} &= \delta a_{sn} \\
\dot{\delta \theta_s} &= \delta \omega_{sn} \\
\\
\delta p_s &\leftarrow \delta p_s + \delta v_s \Delta t \\
\text{Discrete Case} \qquad \qquad \delta v_s &\leftarrow \delta v_s + \delta v_{sn} \\
\delta \theta_s &\leftarrow \delta \theta_s + \delta \theta_{sn}\\
\end{align} 

In [12]:
# Continuous
delta_p_s_dot = delta_v_s
delta_v_s_dot = delta_v_s_n
delta_theta_s_dot = delta_theta_s_n  # not defined, das muss ich noch ausarbeiten

delta_x_s_dot = sym.Matrix.vstack(delta_p_s_dot, delta_v_s_dot, delta_theta_s_dot)

# Discrete
f_delta_p_s = delta_p_s + delta_v_s * dt
f_delta_v_s = delta_v_s + delta_v_s_n
f_delta_theta_s = delta_theta_s + delta_theta_s_n

f_delta_x_s = sym.Matrix.vstack(f_delta_p_s, f_delta_v_s, f_delta_theta_s)
# f_delta_x_s

## Combined State

\begin{align*}
\delta x = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b & \delta p_s & \delta v_s & \delta \theta_i & \delta p_{i + 1} & \delta v_{i + 1} & \delta \theta_{i + 1} & \cdots \end{bmatrix}^\top \\
\end{align*}


In [13]:
delta_x = sym.Matrix.vstack(delta_x_IMU, delta_x_s)
delta_x_n = sym.Matrix.vstack(delta_x_n_IMU, delta_x_s_n)
delta_x_dot = sym.Matrix.vstack(delta_x_IMU_dot, delta_x_s_dot)
f_delta_x = sym.Matrix.vstack(f_delta_x_IMU, f_delta_x_s)

f_delta_x

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢         ⎛                                                                    ↪
⎢\Delta t⋅⎝\delta{\theta}_{x}⋅((-\tilde{a}_{y} + a_{b,y})⋅(-2⋅q_{w}⋅q_{y} - 2⋅ ↪
⎢                                                                              ↪
⎢         ⎛                   ⎛                                                ↪
⎢\Delta t⋅⎝\delta{\theta}_{x}⋅⎝(-\tilde{a}_{y} + a_{b,y})⋅(2⋅q_{w}⋅q_{x} - 2⋅q ↪
⎢                                                                              ↪
⎢         ⎛                 

---

# True State

The true state is mostly unused except for small steps like constucting the measurement jacobian $H_{\delta x}$

In [14]:
p_true = p + delta_p
v_true = v + delta_v
q_true = left_quat_matrix(q) * quat_from_axis_angle(delta_theta)
a_b_true = a_b + delta_a_b
omega_b_true = omega_b + delta_omega_b

x_IMU_true = sym.Matrix.vstack(p_true, v_true, q_true, a_b_true, omega_b_true)

p_s_true = p_s + delta_p_s
v_s_true = v_s + delta_v_s
q_s_true = left_quat_matrix(q_s) * quat_from_axis_angle(delta_theta_s)

x_s_true = sym.Matrix.vstack(p_s_true, v_s_true, q_s_true)

x_true = sym.Matrix.vstack(x_IMU_true, x_s_true)

x_IMU_true, x_s_true

⎛⎡                                       \delta{p}_{x} + p_{x}                 ↪
⎜⎢                                                                             ↪
⎜⎢                                       \delta{p}_{y} + p_{y}                 ↪
⎜⎢                                                                             ↪
⎜⎢                                       \delta{p}_{z} + p_{z}                 ↪
⎜⎢                                                                             ↪
⎜⎢                                       \delta{v}_{x} + v_{x}                 ↪
⎜⎢                                                                             ↪
⎜⎢                                       \delta{v}_{y} + v_{y}                 ↪
⎜⎢                                                                             ↪
⎜⎢                                       \delta{v}_{z} + v_{z}                 ↪
⎜⎢                                                                             ↪
⎜⎢-0.5⋅\delta{\theta}_{x}⋅q_

## Kinematics



---

# Measurements

**IMU**

$$
g^h = \mathbf{R}^{\{q\}}g + a_b
$$

**Face Detection**

$$
\begin{align}
q_s^h &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_s \\
p_s^h &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_s - p) - \Delta p_{I \to c}) \\
\\
\text{with} \qquad\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\tfrac{102°}{2}\right) & \displaystyle \sin\left(-\tfrac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top
\end{align}
$$


In [15]:
# IMU
g_h = R(q).T * gravity_vector_sym + a_b

# face detection
q_h_s = R(IMU_cam_rotation) * (R(q) * (p_s - p) - IMU_cam_translation)
p_h_s = left_quat_matrix(quat_inv(left_quat_matrix(q) * IMU_cam_rotation)) * q_s

## Fake Measurements

In an effort to bound the drift of the IMU and therefore also the global position and velocity values which are currently unobservable with the measurements we have, we introduce a fake measurement. In this case, we periodically "fake measure" the IMUs position to be in the global origin (or the global origin + 1 meter off the ground). While this is not accurate, it might help reduce the drift and lead to a better observability of the system.

$$p^h = p$$


In [16]:
p_h = p

While we do not have the yaw measurement available, we can fake its availability by taking the yaw from the optitrack reading of the IMUs orientation. To extract yaw from a quaternion, we need to calculate
$$\psi^h = \text{atan2}{\left(2 (q_{w} q_{z} - q_{x} q_{y}),1 - 2 (q_{x}^{2} + q_{z}^{2} )\right)}$$
We can do this operation on both the orientation quaternion measured by the Optitrack system as well as the orientation predicted by the filter.


In [17]:
psi_h = sym.Matrix(
    [sym.atan2(2 * (q[0] * q[3] - q[1] * q[2]), 1 - 2 * (q[1] ** 2 + q[3] ** 2))]
)

## Combined

In [18]:
h = sym.Matrix.vstack(g_h, p_h_s, q_h_s)  # without fake measurements
# h = sym.Matrix.vstack(g_h, p_h_s, q_h_s, p_h, psi_h) # with fake measurements
h = sym.simplify(h)
h

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           

---

# Observability (discrete time)

## Weak Observability of Error State

Since the above calculation takes a long time, I will use the second option of showing observability.

$$
\mathcal{O}_{\text{ESEKF}} = \begin{bmatrix} H_{\delta x} \\ H_{\delta x} F_{\delta x} \\ H_{\delta x} F_{\delta x}^2 \\ \vdots \\ H_{\delta x} F_{\delta x}^{n - 1} \end{bmatrix}
$$

With

$$
\begin{align}
H_{\delta x} &= \frac{\partial h(x \oplus \delta x)}{\partial (x \oplus \delta x)} \cdot \frac{\partial (x \oplus \delta x)}{\partial \delta x} = \frac{\partial h}{\partial x_t} \frac{\partial x_t}{\partial \delta x} \\
F_{\delta x} &= \frac{\partial f_{\delta x}(\delta x)}{\partial \delta x}
\end{align}
$$

In an Error-State Kalman Filter (ESEKF), the nominal state $x$ absorbs the large non-linear trajectory, keeping the true error $\delta x$ close to zero. Because measurements arrive frequently, the error never grows large. Furthermore, after every measurement update, the estimated error is injected back into the nominal state and reset to zero ($\mathbb{E}[\delta x] = \mathbf{0}$). Thus, $\delta x = \mathbf{0}$ is the exact point around which the filter operates.


### Randomizes Substitution Setup


In [19]:
def build_substitution_dict(H, F, seed):
    np.random.seed(seed)
    all_symbols = H.free_symbols | F.free_symbols

    # hacky solution, but creating "fake quaternions",
    # only to make the calculations using these values more realistic
    # while keeping their random nature for randomized substitution
    q1_unnormed = [np.random.uniform(-1, 1) for _ in range(4)]
    q2_unnormed = [np.random.uniform(-1, 1) for _ in range(4)]
    q1 = q1_unnormed / np.linalg.norm(q1_unnormed)
    q2 = q2_unnormed / np.linalg.norm(q2_unnormed)
    quat_test_dict = {
        "q_w": q1[0],
        "q_x": q1[1],
        "q_y": q1[2],
        "q_z": q1[3],  # IMU quaternion
        "q_{i,w}": q2[0],
        "q_{i,x}": q2[1],
        "q_{i,y}": q2[2],
        "q_{i,z}": q2[3],  # speaker quaternion
    }

    substitution_dict = {}

    for s in all_symbols:
        if s in subs_fixed_values:
            substitution_dict[s] = subs_fixed_values[s]
        elif s.name in quat_test_dict:
            substitution_dict[s] = quat_test_dict[s.name]
        else:
            substitution_dict[s] = np.random.uniform(-2, 2)

    return substitution_dict

### Observability Matrix construction


In [20]:
def build_obs_matrix(n_states, H_delta_x, F_delta_x, seed=10):
    substitution_dict = build_substitution_dict(H_delta_x, F_delta_x, seed)

    H_sub = np.array((H_delta_x.xreplace(substitution_dict)).tolist(), dtype=np.float64)
    F_sub = np.array((F_delta_x.xreplace(substitution_dict)).tolist(), dtype=np.float64)

    obs_matrix = H_sub
    HF_k = H_sub
    for k in range(n_states):
        HF_k = HF_k @ F_sub
        obs_matrix = np.vstack((obs_matrix, HF_k))

    return obs_matrix

### Rank and Nullspace calculation

We determine rank and nullspace by substituting once. This is sufficient because of the Schwartz Zippel Lemma. Nevertheless, we will also test it for multiple substitution disctionaries just to be safe.


In [21]:
def visualize_nullspace(nullspace, threshold):
    if nullspace.shape[1] <= 1:
        cleaned_nullspace = nullspace
    else:  # RREF
        cleaned_nullspace, _ = sym.Matrix(nullspace.T).rref()
        cleaned_nullspace = np.array(cleaned_nullspace, dtype=float).T

    cleaned_nullspace = np.where(
        np.abs(cleaned_nullspace) < threshold, 0, np.round(cleaned_nullspace, 4)
    )

    # for symbolic visualization
    state_col_mat = np.array(delta_x).reshape(-1, 1)
    nullspace_sign = np.sign(cleaned_nullspace).astype(int)
    symbolic_nullspace = state_col_mat * nullspace_sign

    latex_nullspace = sym.latex(sym.Matrix(nullspace))
    latex_cleaned = sym.latex(sym.Matrix(cleaned_nullspace))
    latex_symbolic = sym.latex(sym.Matrix(symbolic_nullspace))

    display(
        Math(
            f"""
    \\begin{{matrix}}
    \\text{{Column Vector{"" if (nullspace.shape[1] <= 1) else "s"} spanning Nullspace}} & \\quad & \\text{{RREF of Nullspace}} & \\quad & \\text{{Symbolic State Nullspace}} \\\\
    \\\\ {latex_nullspace} & & {latex_cleaned} & & {latex_symbolic}
    \\end{{matrix}}
    """
        )
    )

In [22]:
# shortcut to change, which measurements are included for testing
# g_h, q_h_s, p_h_s, p_h, psi_h
h = sym.Matrix.vstack(g_h, q_h_s, p_h_s, p_h, psi_h)

n_states = delta_x.shape[0]

# jacobians
H_delta_x = h.jacobian(x) * x_true.jacobian(delta_x)
F_delta_x = f_delta_x.jacobian(delta_x)
# Substitute zero error (see markdown explanation)
H_delta_x = H_delta_x.subs({var: 0 for var in delta_x})
F_delta_x = F_delta_x.subs({var: 0 for var in delta_x})

obs_matrix = build_obs_matrix(n_states, H_delta_x, F_delta_x)

U, S, Vh = np.linalg.svd(obs_matrix, full_matrices=False)
threshold = (
    max(obs_matrix.shape) * np.finfo(obs_matrix.dtype).eps * S[0]
)  # threshold = 4.58930124362761e-12

# rank
rank = int(np.sum(S > threshold))
print(
    f"rank: {rank} / {n_states} | {n_states - rank} unobservable state{"" if (n_states - rank == 1) else "s"}"
)

# nullspace
if rank < n_states:
    nullspace = Vh[rank:].T

    visualize_nullspace(nullspace, threshold)

rank: 24 / 24 | 0 unobservable states


## Testing stability of randomized substitution

We create several different versions of the substitution dictionary and run the code on all of them to see if the result is the same for all.


In [23]:
%%script true
timer = Timer()

n_states = delta_x.shape[0]
H_delta_x, F_delta_x = build_jacobians()

for i in range(10,30):
    obs_matrix = build_obs_matrix(n_states, H_delta_x, F_delta_x, i)

    U, S, Vh = np.linalg.svd(obs_matrix, full_matrices=False)
    threshold = max(obs_matrix.shape) * np.finfo(obs_matrix.dtype).eps * S[0] # threshold = 4.58930124362761e-12

    # rank
    rank = int(np.sum(S > threshold))
    print(f"final rank: {rank} / {n_states} | {n_states - rank} unobservable state{"" if (n_states - rank == 1) else "s"} | {timer.check()}")

    # nullspace
    if rank < n_states:
        nullspace = Vh[rank:].T

        visualize_nullspace(nullspace, threshold)

---

# Observability (continuous time)

In [24]:
# h = sym.Matrix.vstack(g_h, q_h_s, p_h_s, p_h, psi_h)
h = sym.Matrix.vstack(g_h, q_h_s, p_h_s, p_h, psi_h)

n_states = delta_x.shape[0]

# jacobians
H_delta_x = h.jacobian(x) * x_true.jacobian(delta_x)
F_delta_x = delta_x_dot.jacobian(delta_x)
# Substitute zero error (see markdown explanation)
H_delta_x = H_delta_x.subs({var: 0 for var in delta_x})
F_delta_x = F_delta_x.subs({var: 0 for var in delta_x})

obs_matrix = build_obs_matrix(n_states, H_delta_x, F_delta_x)


U, S, Vh = np.linalg.svd(obs_matrix, full_matrices=False)
threshold = (
    max(obs_matrix.shape) * np.finfo(obs_matrix.dtype).eps * S[0]
)  # threshold = 4.58930124362761e-12

# rank
rank = int(np.sum(S > threshold))
print(
    f"rank: {rank} / {n_states} | {n_states - rank} unobservable state{"" if (n_states - rank == 1) else "s"}"
)

# nullspace
if rank < n_states:
    nullspace = Vh[rank:].T

    visualize_nullspace(nullspace, threshold)

rank: 24 / 24 | 0 unobservable states


---

# Jacobian Construction and dill Export

In [25]:
# hier könnte ich dann etwas damit spielen, wie ich später die Matrizen und kinematics Funktionen exportieren möchte

# - exportieren (in obsidian gesammelt)
#   - ✔ nominal state kinematics function vector $f$
#   - ✔ error state kinematics jacobian $F_{\delta x}{}$
#   - ✔ error noise covariance $Q{}$
#   - ✔ error noise jacobian $J$
#   - ✔ measurement jacobian $H$
#   - ✔ measurement noise $V$
#   - ✔ measurement function $h$
#   - vielleicht auch actually wie so quaternion multiplication und die umwandlung von $\delta \theta$ zu $q{}$ ($q\{\delta \theta\}{}$)läuft.


# def assert_inputs_match(expr, input_symbols, name):
#     missing = expr.free_symbols - set(input_symbols)
#     display(expr.free_symbols)
#     display(input_symbols)
#     print(missing)
#     if missing:
#         raise ValueError(f"CRITICAL: {name} missing symbols in lambdify: {missing}")


def assert_inputs_match(expr, input_symbols, name):
    input_set = set(input_symbols)

    # in expr, but not lambdified -> would become a global
    missing = expr.free_symbols - input_set
    # lambdified, but not in expr -> suspicious
    unused = input_set - expr.free_symbols

    if missing:
        raise ValueError(
            f"CRITICAL: {name} forgot to include these symbols in lambdify: {missing}"
        )
    if unused:
        raise ValueError(
            f"WARNING-level strict: {name} has unused lambdify inputs "
            f"(not in {name} expression): {unused}"
        )


# nominal state kinematics function $f$
# ---
# I need to set the order of inputs that will later be used when calling the function.
# In this case I decide on f(*x, *u, gravity, dt)
f_x_IMU_input_symbols = list(x_IMU) + list(u_tilde) + [gravity] + [dt]
assert_inputs_match(f_x_IMU, f_x_IMU_input_symbols, "f_x_IMU")
f_x_IMU_lambdified = sym.lambdify(f_x_IMU_input_symbols, f_x_IMU, "numpy")

# error state kinematics jacobian $F_{\delta x}{}$
# ---
F_delta_x = f_delta_x_IMU.jacobian(delta_x_IMU)
F_delta_x = F_delta_x.subs({var: 0 for var in delta_x_IMU})
F_delta_x = sym.simplify(F_delta_x)
F_delta_x_input_symbols = list(q) + list(a_b) + list(omega_b) + list(u_tilde) + [dt]
assert_inputs_match(F_delta_x, F_delta_x_input_symbols, "F_delta_x")
F_delta_x_lambdified = sym.lambdify(F_delta_x_input_symbols, F_delta_x, "numpy")

# true state in order to perform error injection later
# ---
x_IMU_true_symbols = list(x_IMU) + list(delta_x_IMU)
assert_inputs_match(x_IMU_true, x_IMU_true_symbols, "x_IMU_true")
x_IMU_true_lambdified = sym.lambdify(x_IMU_true_symbols, x_IMU_true, "numpy")


# error noise jacobian $J$
# ---
# jacobian of error function with respect to pertubation impulses $j$
# its always some form of almost identity matrix because of the addititve error
J = f_delta_x_IMU.jacobian(delta_x_n_IMU)
J_lambdified = sym.lambdify([], J, "numpy")

# error noise covariance $Q{}$
# ---
# only temporary implementation approach, this will later need to be more complex
# as it will not just have purely diagonal elements but the diagonal is comprised of covariancematrices itself
Q_diagonal = sym.Matrix([])
for i in range(len(delta_x_n_IMU)):
    Q_diagonal = sym.Matrix.vstack(Q_diagonal, sym.Matrix([sym.Symbol(f"Q_{i}")]))
Q = sym.diag(*Q_diagonal)
Q_lambdified = sym.lambdify(Q_diagonal, Q, "numpy")

# measurement function $h$ ✔
# ---
# similar to the observability tests, I need to decide which measurements to include
h = sym.Matrix.vstack(g_h)  # ,p_h, psi_h
h_input_symbols = list(q) + list(a_b) + [gravity]
assert_inputs_match(h, h_input_symbols, "h")
h_lambdified = sym.lambdify(h_input_symbols, h, "numpy")

# measurement jacobian $H$
H_delta_x = h.jacobian(x_IMU) * x_IMU_true.jacobian(delta_x_IMU)
H_delta_x = H_delta_x.subs({var: 0 for var in delta_x_IMU})
H_delta_x = sym.simplify(H_delta_x)
H_delta_x_input_symbols = list(q) + [gravity]
assert_inputs_match(H_delta_x, H_delta_x_input_symbols, "H_delta_x")
H_delta_x_lambdified = sym.lambdify(H_delta_x_input_symbols, H_delta_x, "numpy")

# measurement noise $V$
# ---
# Same as Q, only temporary because I haven't yet decided on the correct conventions as well
V_diagonal = sym.Matrix([])
for i in range(len(h)):
    V_diagonal = sym.Matrix.vstack(V_diagonal, sym.Matrix([sym.Symbol(f"V_{i}")]))
V = sym.diag(*V_diagonal)
V_lambdified = sym.lambdify(V_diagonal, V, "numpy")

In [26]:
with open("filter_funcs_IMU_only.pkl", "wb") as f:
    dill.dump(
        {
            "f_x_IMU": f_x_IMU_lambdified,
            "F_delta_x": F_delta_x_lambdified,
            "x_IMU_true": x_IMU_true_lambdified,
            "J": J_lambdified,
            "Q": Q_lambdified,
            "h": h_lambdified,
            "H_delta_x": H_delta_x_lambdified,
            "V": V_lambdified,
        },
        f,
    )

---

# debugging and test stuff


## yaw insert testing

I am trying to derive the correct atan2 yaw extraction


In [27]:
forward_directions = {
    "forward_y_pos": [0, 1, 0],
    "forward_y_neg": [0, -1, 0],
    "forward_x_pos": [1, 0, 0],
    "forward_x_neg": [-1, 0, 0],
}

q_test = sym.Matrix(sym.symbols("q_w, q_x, q_y, q_z"))

rotated_forward_directions = {
    name: R(q_test) * sym.Matrix(direction)
    for name, direction in forward_directions.items()
}

yaw_calculation_dict = {
    name: sym.atan2(-direction[0], direction[1])
    for name, direction in rotated_forward_directions.items()
}

for name, yaw in yaw_calculation_dict.items():
    display(yaw)

In [28]:
forward_y_pos = sym.Matrix([0, 1, 0])
forward_x_pos = sym.Matrix([1, 0, 0])

q_test = sym.Matrix(sym.symbols("q_w, q_x, q_y, q_z"))

rotated_forward_y_pos = R(q_test) * forward_y_pos
rotated_forward_x_pos = R(q_test) * forward_x_pos

y_yaw = sym.atan2(-rotated_forward_y_pos[0], rotated_forward_y_pos[1])
x_yaw = sym.atan2(rotated_forward_x_pos[1], rotated_forward_x_pos[0])

display(y_yaw)
display(x_yaw)

In [29]:
Q = sym.diag(*sym.Matrix(sym.symbols("V, \\Theta, A, \\Omega, v_s, \\Theta_i")))
J = sym.diag(sym.Matrix([0, 1]), 1, 1, sym.Matrix([1, 0]), 1, 1)
J * Q * J.T

⎡0  0    0     0    0     0  0      0   ⎤
⎢                                       ⎥
⎢0  V    0     0    0     0  0      0   ⎥
⎢                                       ⎥
⎢0  0  \Theta  0    0     0  0      0   ⎥
⎢                                       ⎥
⎢0  0    0     A    0     0  0      0   ⎥
⎢                                       ⎥
⎢0  0    0     0  \Omega  0  0      0   ⎥
⎢                                       ⎥
⎢0  0    0     0    0     0  0      0   ⎥
⎢                                       ⎥
⎢0  0    0     0    0     0  vₛ     0   ⎥
⎢                                       ⎥
⎣0  0    0     0    0     0  0   \Thetaᵢ⎦